# Stiffness Matrix Analysis

This notebook is to verify the stiffness matrix of mirror from vendor, which defines the relationship between the actuator force and step in a small linear range to be used in the closed-loop control.
In the bump test, 78 actuators applies the positive and negative bumping under the closed-loop control.
The applied force of axial actuator is 10 N and the applied force of tangent link is 100 N.
Since the test is perfomed under the closed-loop control, the force balance system would be involved in and the data analysis would be tricky compared with the same test under the open-loop control.

## Summary

## Import Modules

This notebook needs to setup the **ts_m2com**, **ts_aos_utilsts**, **ts_config_mttcs** under the **notebooks/.user_setups**, which depend on the **ts_tcpip** and **ts_xml**.

In [ ]:
%matplotlib inline
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from astropy.time import Time

from lsst.ts.m2com import read_yaml_file, get_config_dir, NUM_ACTUATOR, NUM_TANGENT_LINK
from lsst.ts.aos.utils import DiagnosticsM2, EfdName

## Declaration of User-defined Functions

In [ ]:
def get_stiff_m2() -> np.ndarray:
    """Get the M2 stiffness 78x78 matrix.

    Return
    ------
    `numpy.ndarray`
        M2 stiffness matrix.
    """
    data = read_yaml_file(get_config_dir() / "harrisLUT" / "stiff_matrix_m2.yaml")["stiff"]
    return np.array(data)

In [ ]:
async def get_efd_data(
    diagnostics_m2: DiagnosticsM2,
    time_start: Time,
    time_end: Time,
    hardpoints,
) -> tuple[np.ndarray, np.ndarray, dict, dict]:
    """Get the EFD data.

    Parameters
    ----------
    diagnostics_m2 : `lsst.ts.aos.utils.DiagnosticsM2`
        M2 diagnostics instance.
    time_start : `astropy.time.core.Time`
        Start time.
    time_end : `astropy.time.core.Time`
        End time.
    hardpoints : `list` [`int]
        Ordered 0-based six hardpoints. The first three are the axial
        actuators and the latters are the tangent links.

    Returns
    -------
    data_steps_axial : `numpy.ndarray`
        Data of the steps of axial actuators.
    data_steps_tangent : `numpy.ndarray`
        Data of the steps of tangent links.
    data_force_axial : `dict`
        Data of the axial force.
    data_force_tangent : `dict`
        Data of the tangent force.
    """
    data_steps_axial, _ = await diagnostics_m2.get_data_step_axial(
        time_start,
        time_end,
        True,
    )
    data_steps_tangent, _ = await diagnostics_m2.get_data_step_tangent(
        time_start,
        time_end,
        True,
    )
    data_force_axial, data_force_tangent = await diagnostics_m2.get_data_force(
        time_start,
        time_end,
        hardpoints,
    )

    return data_steps_axial, data_steps_tangent, data_force_axial, data_force_tangent

In [ ]:
def get_indeces_bump(data_force_axial: dict, data_force_tangent: dict) -> list[int]:
    """Get the indeces of bumping.

    Parameters
    ----------
    data_force_axial : `dict`
        Data of the axial force.
    data_force_tangent : `dict`
        Data of the tangent force.

    Returns
    -------
    `list`
        Indeces of the bumping.
    """

    len_data = min(len(data_force_axial["applied"]), len(data_force_tangent["applied"]))

    indeces = list()
    for idx in range(1, len_data):
        if (
            np.sum(np.abs(data_force_axial["applied"][idx])) != 0.0 and
            np.sum(np.abs(data_force_axial["applied"][idx - 1])) == 0.0
        ):
           indeces.append(idx)

        # Note I can do this because I know the bumping is one-by-one
        if (
            np.sum(np.abs(data_force_tangent["applied"][idx])) != 0.0 and
            np.sum(np.abs(data_force_tangent["applied"][idx - 1])) == 0.0
        ):
           indeces.append(idx)

    return indeces

In [ ]:
def calc_delta_force(
    stiffness: np.ndarray,
    data_steps_axial: np.ndarray,
    data_steps_tangent: np.ndarray,
    data_force_axial: dict,
    data_force_tangent: dict,
    idx: int,
    offset: int,
) -> np.ndarray:
    """Calculate the theoretical delta force based on the stiffness matrix and the
    delta actuator steps.

    Parameters
    ----------
    stiffness : `numpy.ndarray`
        78x78 stiffness matrix.
    data_steps_axial : `numpy.ndarray`
        Data of the steps of axial actuators.
    data_steps_tangent : `numpy.ndarray`
        Data of the steps of tangent links.
    data_force_axial : `dict`
        Data of the axial force.
    data_force_tangent : `dict`
        Data of the tangent force.
    idx : `int`
        Index of the data.
    offset : `int`
        Index offset.

    Returns
    -------
    delta_force_theoretical: `numpy.ndarray`
        Theoretical delta force in N.
    delta_force_measured : `numpy.ndarray`
        Measured delta force in N.
    delta_step_change : `numpy.ndarray`
        Delta step change.
    """

    # Put the -1 here to get the data that is just before the bumping
    idx_start = idx - 1
    idx_end = idx + offset

    # Calculate the delta force based on the stiffness matrix
    delta_step_change = np.append(
        data_steps_axial[idx_end] - data_steps_axial[idx_start],
        data_steps_tangent[idx_end] - data_steps_tangent[idx_start],
    )
    delta_force_theoretical = stiffness.dot(delta_step_change)

    # Calculate the measured delta force
    delta_force_measured = np.append(
        data_force_axial["measured"][idx_end] - data_force_axial["measured"][idx_start],
        data_force_tangent["measured"][idx_end] - data_force_tangent["measured"][idx_start],
    )

    return delta_force_theoretical, delta_force_measured, delta_step_change

In [ ]:
def print_info(
    stiffness: np.ndarray,
    data_steps_axial: np.ndarray,
    data_steps_tangent: np.ndarray,
    data_force_axial: dict,
    data_force_tangent: dict,
    idx: int,
    offset: int,
) -> None:
    """Print the information.

    Parameters
    ----------
    stiffness : `numpy.ndarray`
        78x78 stiffness matrix.
    data_steps_axial : `numpy.ndarray`
        Data of the steps of axial actuators.
    data_steps_tangent : `numpy.ndarray`
        Data of the steps of tangent links.
    data_force_axial : `dict`
        Data of the axial force.
    data_force_tangent : `dict`
        Data of the tangent force.
    idx : `int`
        Index of the data.
    offset : `int`
        Index offset.
    """

    force_applied = np.append(data_force_axial["applied"][idx], data_force_tangent["applied"][idx])
    idx_actuator_bump = np.where(force_applied != 0)[0][0]
    print(f"Apply {force_applied[idx_actuator_bump]} N to the actuator {idx_actuator_bump}.")

    delta_force_theoretical, delta_force_measured, delta_step_change = calc_delta_force(
        stiffness,
        data_steps_axial,
        data_steps_tangent,
        data_force_axial,
        data_force_tangent,
        idx,
        offset,
    )

    print(f"Measured delta force of the actuator {idx_actuator_bump} is "
          f"{delta_force_measured[idx_actuator_bump]} N."
    )
    print(f"Calculated delta force of the actuator {idx_actuator_bump} is "
          f"{delta_force_theoretical[idx_actuator_bump]} N."
    )

    step = delta_step_change[idx_actuator_bump]
    print(f"Measured delta step of the actuator {idx_actuator_bump} is {step}.")

    if (step == 0):
        print("Note this is the hardpoint and there is no step change.")

    num_axial = NUM_ACTUATOR - NUM_TANGENT_LINK
    difference = delta_force_theoretical - delta_force_measured
    print("Check 72 axial actuators and the maximum difference between the theory and "
          f"measurement is {np.max(np.abs(difference[:num_axial]))} N."
    )
    print("Check 6 tangent links and the maximum difference between the theory and "
          f"measurement is {np.max(np.abs(difference[num_axial:]))} N."
    )

    if (idx_actuator_bump >= num_axial):
        print("The tangent link is bumped. Check the steps and forces of tangent links")
        print(f"Delta steps: {delta_step_change[num_axial:]}")
        print(f"Delta force (measurement): {delta_force_measured[num_axial:]} N")
        print(f"Delta force (theory): {delta_force_theoretical[num_axial:]} N")
        print(f"Difference between the theory and measurement: {difference[num_axial:]} N")

In [ ]:
def plot_tagnent_links_data(
    stiffness: np.ndarray,
    data_steps_axial: np.ndarray,
    data_steps_tangent: np.ndarray,
    data_force_axial: dict,
    data_force_tangent: dict,
    idx: int,
    max_offset: int,
) -> None:
    """Plot the tangent link data.

    Parameters
    ----------
    stiffness : `numpy.ndarray`
        78x78 stiffness matrix.
    data_steps_axial : `numpy.ndarray`
        Data of the steps of axial actuators.
    data_steps_tangent : `numpy.ndarray`
        Data of the steps of tangent links.
    data_force_axial : `dict`
        Data of the axial force.
    data_force_tangent : `dict`
        Data of the tangent force.
    idx : `int`
        Index of the data.
    max_offset : `int`
        Maximum of the index offset.
    """

    data_steps_tangent_data = np.zeros((max_offset, NUM_TANGENT_LINK))
    data_force_tangent_data_theoretical = np.zeros((max_offset, NUM_TANGENT_LINK))
    data_force_tangent_data_measured = np.zeros((max_offset, NUM_TANGENT_LINK))
    for ii in range(0, max_offset):
        delta_force_theoretical, delta_force_measured, delta_step_change = calc_delta_force(
            stiffness,
            data_steps_axial,
            data_steps_tangent,
            data_force_axial,
            data_force_tangent,
            idx,
            ii,
        )
    
        data_steps_tangent_data[ii, :] = delta_step_change[-NUM_TANGENT_LINK:]
        data_force_tangent_data_theoretical[ii, :] = delta_force_theoretical[-NUM_TANGENT_LINK:]
        data_force_tangent_data_measured[ii, :] = delta_force_measured[-NUM_TANGENT_LINK:]

    fig, ax = plt.subplots(2, 3)
    for idx in range(NUM_TANGENT_LINK):
        row = 0 if idx < 3 else 1
        col = idx % 3
        ax[row, col].plot(
            data_steps_tangent_data[:, idx],
            data_force_tangent_data_theoretical[:, idx],
            "bx",
        )
        ax[row, col].plot(
            data_steps_tangent_data[:, idx],
            data_force_tangent_data_measured[:, idx],
            "ro",
        )
        ax[row, col].set_title(f"A{idx+1}")
    
    ax[0, 0].set_ylabel("Delta Force (N)")
    ax[1, 0].set_ylabel("Delta Force (N)")
    ax[1, 1].set_xlabel("Delta Step")
    
    # fig.suptitle("Position")
    fig.tight_layout()
    fig.legend(["Theory", "Measurement"], loc="lower right")
    
    plt.show()

## Instantiate the DiagnosticsM2 Class

Notice that the UTC time is used when doing the query.

In [ ]:
time_start = Time("2024-07-15T16:27:42", scale="utc", format="isot")
time_end = Time("2024-07-15T17:08:33", scale="utc", format="isot")

In [ ]:
diagnostics_m2 = DiagnosticsM2(efd_name=EfdName.Usdf)
hardpoints = [5, 15, 25, 73, 75, 77]

In [ ]:
# Query the data from EFD
data_steps_axial, data_steps_tangent, data_force_axial, data_force_tangent = await get_efd_data(
    diagnostics_m2,
    time_start,
    time_end,
    hardpoints,
)

In [ ]:
# Check the length of the queried data. They should be similar.
print(f"Length of data_steps_axial is {len(data_steps_axial)}.")
print(f"Length of data_steps_tangent is {len(data_steps_tangent)}.")
print(f"Length of data_force_axial is {len(data_force_axial['measured'])}.")
print(f"Length of data_force_tangent is {len(data_force_tangent['measured'])}.")

# Check the frequency of data
frequency = len(data_steps_axial) / (time_end - time_start).to_value("s")
print(f"Frequency of telemetry is {frequency} Hz.")

In [ ]:
# Get the indeces of the bumping
indeces = get_indeces_bump(data_force_axial, data_force_tangent)

In [ ]:
# Check the length of indeces, which should be 78 * 2 = 156
print(f"Length of the bumping indeces is {len(indeces)}.")

In [ ]:
# Read the stiffness matrix
stiffness = get_stiff_m2()

In [ ]:
# Print the information

# Put the offset to be 10 (~ 2 sec) to get the stabilized force data.
offset = 10

# Print the some picked axial actuators
for idx in (0, 11):
    print_info(
        stiffness,
        data_steps_axial,
        data_steps_tangent,
        data_force_axial,
        data_force_tangent,
        indeces[idx],
        offset,
    )
    print("\n")

# Print the tangent links
for idx in range(144, 156):
    print_info(
        stiffness,
        data_steps_axial,
        data_steps_tangent,
        data_force_axial,
        data_force_tangent,
        indeces[idx],
        offset,
    )
    print("\n")

In [ ]:
idx = 144

plot_tagnent_links_data(
    stiffness,
    data_steps_axial,
    data_steps_tangent,
    data_force_axial,
    data_force_tangent,
    indeces[idx],
    offset,
)

In [ ]:
plt.close("all")